In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.append('../..')
import data_preprocessing.coil20 as coil20
from importlib import reload
reload(coil20)

In [ ]:
mat, label = coil20.read_data(num_of_class=2, random_shuffle=True, random_choose_class=True)
mat = torch.tensor(mat, dtype=torch.float32)

print(label)

# A = mat @ mat.T

# compute the similarity matrix using cosine similarity
mat_normalized = mat / torch.norm(mat, dim=1).reshape(-1, 1)
A = mat_normalized @ mat_normalized.T

In [ ]:
def loss_without_regularization(A, H, lambda_=None):
    '''
    A: a symmetric matrix of size n x n
    H: a matrix of size n x k
    '''
    assert A.T.equal(A), "A is not symmetric"
    
    return torch.norm(A - H @ H.T) ** 2

def loss_with_regularization_optimized(A, H, lambda_):
    '''
    A: a symmetric matrix of size n x n
    H: a matrix of size n x k
    lambda_: a scalar
    '''
    assert A.T.equal(A), "A is not symmetric"
    
    loss = torch.norm(A - H @ H.T) ** 2
    
    # Vectorized computation of regularization loss
    diff = H[:, None, :] - H[None, :, :]
    regularization_loss = (A.unsqueeze(-1) * diff**2).sum()
    regularization_loss *= lambda_
    
    return loss + regularization_loss

In [ ]:
import torch.optim as optim

def Gradient_Descent(A, H, lamda, 
                     loss_fn,
                     lr,
                     max_iter=100, tol=1e-4, print_every=100):
    '''
    A: a symmetric matrix of size n x n
    H: a matrix of size n x k
    lambda_: a scalar
    '''
    adam = optim.Adam([H], lr=lr)
    
    assert H.requires_grad, "H is not set to require grad"
    if H.grad is not None:
        H.grad.zero_()
        
    loss_last = 0
    for i in range(max_iter):
        loss = loss_fn(A, H, lamda)
        loss.backward()
        with torch.no_grad():
            adam.step()
            adam.zero_grad()
        if i % print_every == 0:
            print(f'iteration {i}, loss {loss.item()}')
        if abs(loss - loss_last) < tol:
            break
        else:
            loss_last = loss
    return H, loss

In [ ]:
N = A.shape[0]
k = 2
H = torch.randn(N, k)
H.requires_grad = True

In [ ]:
rst_1 = Gradient_Descent(A=A, H=H, lamda=1, 
                         loss_fn=loss_with_regularization_optimized, 
                         lr=1e-5, max_iter=100000,
                         tol=1e-6, print_every=500)

In [ ]:
rst_1 = Gradient_Descent(A=A, H=H, lamda=0, 
                         loss_fn=loss_without_regularization, 
                         lr=1e-6, max_iter=100000,
                         tol=1e-8, print_every=10000)

In [ ]:
pred_labels = torch.argmax(rst_1[0], dim=1).detach().numpy()

import eval_metrics.ac

eval_metrics.ac.ac(label, pred_labels)

In [ ]:
plt.imshow(rst_1[0].detach().numpy()[:100])

In [ ]:
# rst_1 = Gradient_Descent(A=A, H=H, lamda=0, 
#                          loss_fn=loss_without_regularization, 
#                          lr=1e-6, max_iter=100000,
#                          tol=1e-4, print_every=1000)
# k=2, 100000 iterations, acc = 0.9
# loss 76